# 面试题：怎样检测多 Agent 写冲突？

回答要点：用版本号或 ETag 的 compare-and-swap 将基于旧快照的写入变成明确冲突；服务端检查资源当前版本和业务不变量。CAS 失败后先回读，再重新验证目标与权限；不自动合并不可解释的写。下面以库存 Agent 和促销 Agent 修改同一商品可售量为例。

## 真实案例

商品 P9 初始库存为 10、版本 1；六次写入包含两个 Agent 基于同一旧版本的更新和一次回读后重试。

## 基线

基线采用最后写入覆盖。

## 结果解读

手写 CAS 输出 expected version、actual version 和冲突。

## 失败案例

促销 Agent 使用过期版本，不能覆盖库存 Agent 已经提交的扣减。

In [1]:
writes = [{'id':'W1','agent':'inventory','expected':1,'delta':-2}, {'id':'W2','agent':'promo','expected':1,'delta':-3}, {'id':'W3','agent':'promo','expected':2,'delta':-3}, {'id':'W4','agent':'inventory','expected':3,'delta':-1}, {'id':'W5','agent':'promo','expected':4,'delta':5}, {'id':'W6','agent':'inventory','expected':5,'delta':-4}]  # 构造六个带预期版本的库存写请求。
print('写入请求:', writes)  # 输出调用方看到的快照版本和库存变化。
print('初始资源: P9 库存=10, version=1')  # 输出权威资源初始状态。

写入请求: [{'id': 'W1', 'agent': 'inventory', 'expected': 1, 'delta': -2}, {'id': 'W2', 'agent': 'promo', 'expected': 1, 'delta': -3}, {'id': 'W3', 'agent': 'promo', 'expected': 2, 'delta': -3}, {'id': 'W4', 'agent': 'inventory', 'expected': 3, 'delta': -1}, {'id': 'W5', 'agent': 'promo', 'expected': 4, 'delta': 5}, {'id': 'W6', 'agent': 'inventory', 'expected': 5, 'delta': -4}]
初始资源: P9 库存=10, version=1


In [2]:
unsafe_stock = 10  # 初始化最后写入覆盖基线中的库存。
for row in writes:  # 按到达顺序忽略版本直接应用每次变化。
    unsafe_stock += row['delta']  # 让过期写也覆盖当前资源状态。
print('最后写入覆盖库存:', unsafe_stock)  # 输出没有冲突检测的最终库存。
print('基线无法告诉调用方 W2 是否覆盖了 W1 的更新。')  # 说明 last-write-wins 丢失的因果信息。

最后写入覆盖库存: 2
基线无法告诉调用方 W2 是否覆盖了 W1 的更新。


In [3]:
resource = {'stock':10,'version':1}  # 初始化带版本号的权威库存资源。
def compare_and_swap(row):  # 定义服务端 CAS 写入门禁。
    if row['expected'] != resource['version']:  # 检查调用方快照是否仍等于当前权威版本。
        return 'conflict', dict(resource)  # 返回当前状态供 Agent 回读再提议。
    if resource['stock'] + row['delta'] < 0:  # 检查库存不能小于零的业务不变量。
        return 'rejected_invariant', dict(resource)  # 拒绝即使版本正确但不合法的更新。
    resource['stock'] += row['delta']  # 原子应用通过版本检查的库存变化。
    resource['version'] += 1  # 推进资源版本使旧快照立即失效。
    return 'applied', dict(resource)  # 返回写入后的权威状态。

In [4]:
results = [(row['id'],) + compare_and_swap(row) for row in writes]  # 对六个写请求按顺序运行 CAS。
print('id | CAS 结论 | 权威状态')  # 输出冲突检测结果表标题。
for item in results:  # 遍历每次写的应用或冲突证据。
    print(item[0], item[1], item[2])  # 输出当前写入的服务端结论。
print('最终权威资源:', resource)  # 输出拒绝过期写后的真实库存与版本。

id | CAS 结论 | 权威状态
W1 applied {'stock': 8, 'version': 2}
W2 conflict {'stock': 8, 'version': 2}
W3 applied {'stock': 5, 'version': 3}
W4 applied {'stock': 4, 'version': 4}
W5 applied {'stock': 9, 'version': 5}
W6 applied {'stock': 5, 'version': 6}
最终权威资源: {'stock': 5, 'version': 6}


In [5]:
wrong = '覆盖写入'  # 模拟 W2 在 last-write-wins 系统中的错误行为。
fixed = results[1]  # 读取 W2 在 CAS 下得到的冲突结果。
print('失败案例 W2：基线=', wrong, '，CAS=', fixed)  # 展示过期促销写不会静默覆盖库存扣减。
print('生产差距：跨资源更新需条件更新、事务或 Saga；多实例不能只依赖 Agent 进程内锁，还需审计冲突率。')  # 说明 CAS 的适用边界。

失败案例 W2：基线= 覆盖写入 ，CAS= ('W2', 'conflict', {'stock': 8, 'version': 2})
生产差距：跨资源更新需条件更新、事务或 Saga；多实例不能只依赖 Agent 进程内锁，还需审计冲突率。


In [6]:
assert results[0][1] == 'applied'  # 验证第一个基于当前版本的库存扣减可成功。
assert results[1][1] == 'conflict'  # 验证第二个使用旧版本的促销写会冲突。
assert resource['version'] == 6  # 验证五个合法写入推进了版本。